In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from catboost import CatBoostRegressor


import sklearn
sklearn.set_config(transform_output="pandas")
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_squared_log_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, OrdinalEncoder, TargetEncoder, FunctionTransformer

In [ ]:
df = pd.read_csv('../train.csv')
df=df.drop_duplicates()
df = df.drop('Id', axis=1, errors='ignore')
condition = (df['GrLivArea'] > 4000) & (df['SalePrice'] < 160001)
df=df[~condition]

Q1_price = df['SalePrice'].quantile(0.25)
Q3_price = df['SalePrice'].quantile(0.75)
IQR_price = Q3_price - Q1_price
lower_bound_price = Q1_price - 1.5 * IQR_price
upper_bound_price = Q3_price + 1.5 * IQR_price
    
condition_price = (df['SalePrice'] < lower_bound_price) | (df['SalePrice'] > upper_bound_price)
df = df[~condition_price]

Q1_lot = df['LotArea'].quantile(0.25)
Q3_lot = df['LotArea'].quantile(0.75)
IQR_lot = Q3_lot - Q1_lot
upper_bound_lot = Q3_lot + 3 * IQR_lot

condition_lot = df['LotArea'] > upper_bound_lot
df = df[~condition_lot]


cond_frontage = df['LotFrontage'] > 250
df = df[~cond_frontage]

df['TotalPorchSF'] = (df['OpenPorchSF'] + df['EnclosedPorch'] +
                          df['3SsnPorch'] + df['ScreenPorch'])
cond_porch = df['TotalPorchSF'] > 700
df = df[~cond_porch]
df = df.drop('TotalPorchSF', axis=1)
    

In [367]:
X, y = df.drop('SalePrice', axis=1), df['SalePrice']

y_log = np.log1p(y)

X_train, X_valid, y_train_log, y_valid_log = train_test_split(X, y_log, test_size=0.3, random_state=42)

y_train_original = np.expm1(y_train_log)
y_valid_original = np.expm1(y_valid_log)

In [368]:
cat_features = ['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities',
                'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2',
                'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st',
                'Exterior2nd', 'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
                'Heating', 'CentralAir', 'Electrical', 'Functional', 'GarageType', 'GarageFinish',
                'PavedDrive', 'Fence', 'MiscFeature','SaleType', 'SaleCondition', 'PoolQC', 'ExterQual','ExterCond',
                'BsmtQual', 'BsmtCond','HeatingQC','KitchenQual','FireplaceQu', 'GarageQual', 'GarageCond']
num_features = ['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
                'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
                'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
                'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
                'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'TotRmsAbvGrd',
                'Fireplaces', 'GarageYrBlt', 'GarageCars', 'GarageArea', 'WoodDeckSF',
                'OpenPorchSF', 'EnclosedPorch', 'ScreenPorch', 'PoolArea', 'MoSold', 'YrSold', '3SsnPorch', 'MiscVal' ]

In [369]:
def clean_data(X):
    X=X.copy()
    X['MasVnrType'] = X['MasVnrType'].fillna('None')
    X['FireplaceQu'] = X['FireplaceQu'].fillna('None')
    X['GarageQual'] = X['GarageQual'].fillna('None')
    X['GarageFinish'] = X['GarageFinish'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None')
    X['GarageCond'] = X['GarageCond'].fillna('None')

    X['Alley'] = X['Alley'].fillna('None')
    X['PoolQC'] = X['PoolQC'].fillna('None')
    X['GarageType'] = X['GarageType'].fillna('None') 

    X['GarageArea'] = X['GarageArea'].fillna(0) 
    X['MasVnrArea'] = X['MasVnrArea'].fillna(0)

    X['GarageYrBlt'] = X['GarageYrBlt'].fillna(X['YearBuilt'])
    X['GarageAge'] = X['YrSold'] - X['GarageYrBlt']
    X['RemAge'] = X['YrSold'] - X['YearRemodAdd']
    X['HouseAge'] = X['YrSold'] - X['YearBuilt']
    X['IsNew'] = (X['HouseAge'] <= 5).astype(int)
    X['IsOld'] = (X['HouseAge'] >= 70).astype(int)
    X['IsHistoric'] = (X['HouseAge'] >= 100).astype(int)
    X['TotalBath'] = (X['FullBath'] + (0.5 * X['HalfBath']) +  #добавлено
                      X['BsmtFullBath'] + (0.5 * X['BsmtHalfBath']))
    X['HasPool'] = X['PoolArea'].apply(lambda x: 1 if x > 0 else 0)
    X['TotalSF'] = (X['GrLivArea'] + 
                    X['TotalBsmtSF'].fillna(0) + 
                    X['GarageArea'].fillna(0) +
                    X['WoodDeckSF'] + 
                    X['OpenPorchSF'] + 
                    X['EnclosedPorch'] + 
                    X['3SsnPorch'] + 
                    X['ScreenPorch'])
    
    X['TotalBathrooms'] = X['FullBath'] + 0.5*X['HalfBath'] + X['BsmtFullBath'] + 0.5*X['BsmtHalfBath']
    X['TotalRooms'] = X['TotRmsAbvGrd'] + X['BedroomAbvGr']
    X['AreaPerRoom'] = X['GrLivArea'] / (X['TotRmsAbvGrd'] + 1)
    X['QualityScore'] = X['OverallQual'] * X['OverallCond']
    X['IsRenovated'] = (X['YearRemodAdd'] != X['YearBuilt']).astype(int)
    X['YearsSinceRenovation'] = X['YrSold'] - X['YearRemodAdd']
    
    return X

new_num_features = num_features + ['GarageAge', 'RemAge', 'HouseAge', 'TotalSF', 'TotalBath', 
                                   'IsNew', 'IsOld', 'IsHistoric', 'HasPool',
                                   'TotalBathrooms', 'TotalRooms', 'AreaPerRoom', 
                                   'QualityScore', 'IsRenovated', 'YearsSinceRenovation']

def imputer_groupby_Neighborhood(X):
    X = X.copy()
    X['LotFrontage'] = X.groupby('Neighborhood')['LotFrontage'].transform(lambda x: x.fillna(x.median()))
    return X

cleaner = FunctionTransformer(clean_data)
group_imputer = FunctionTransformer(imputer_groupby_Neighborhood)

drop_features = ['MiscFeature', 'Utilities', 'PoolQC', 'Alley', 'Fence',
                 'LowQualFinSF', 'MiscVal', 'HalfBath', 'BsmtHalfBath', 'Street', 'LandSlope'] 

In [370]:
binary_and_low_cardinality = []   
high_cardinality = []       

for col in cat_features:
    n_unique = X_train[col].nunique()
    if n_unique <= 10:
        binary_and_low_cardinality.append(col)
    else:
        high_cardinality.append(col)

my_imputer = ColumnTransformer(
    transformers = [
        ('drop_features', 'drop', drop_features),
        
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler())
        ]), new_num_features),
        
        # Бинарные - OneHot
        ('binary', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), binary_and_low_cardinality),
        
        # Высокая кардинальность - FrequencyEncoder
        ('high_cat_freq', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', TargetEncoder(smooth=20))
        ]), high_cardinality)
    ],
    verbose_feature_names_out=False,
    remainder='drop'
)

preprocessor = Pipeline(
    [
        ('cleaner', cleaner),
        ('group_imputer', group_imputer),
        ('imputer', my_imputer)
    ]
)

In [371]:
preprocessor.fit(X_train, y_train_log)

X_train_processed = preprocessor.transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

In [372]:
# X_train_array = np.array(X_train_processed, dtype=np.float64)
# X_valid_array = np.array(X_valid_processed, dtype=np.float64)


# scaler_lasso = StandardScaler()
# X_train_scaled = scaler_lasso.fit_transform(X_train_array)
# X_valid_scaled = scaler_lasso.transform(X_valid_array)

# # ПРИНУДИТЕЛЬНО КОНВЕРТИРУЕМ В NUMPY
# X_train_scaled = np.asarray(X_train_scaled)
# X_valid_scaled = np.asarray(X_valid_scaled)

# print(f"X_train_scaled тип: {type(X_train_scaled)}")
# print(f"X_train_scaled форма: {X_train_scaled.shape}")

# # Проверяем на бесконечности (теперь должно работать)
# if np.isinf(X_train_scaled).any():
#     X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
#     X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# alpha = 0.005

# lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
# lasso.fit(X_train_scaled, y_train_log.values)

# selected_mask = np.abs(lasso.coef_) > 1e-5
# n_selected = np.sum(selected_mask)
# print(f"Alpha = {alpha}")
# print(f"Отобрано признаков: {n_selected} из {X_train_scaled.shape[1]}")

In [373]:
X_train_array = np.array(X_train_processed, dtype=np.float64)
X_valid_array = np.array(X_valid_processed, dtype=np.float64)


scaler_lasso = StandardScaler()
X_train_scaled = scaler_lasso.fit_transform(X_train_array)
X_valid_scaled = scaler_lasso.transform(X_valid_array)

# ПРИНУДИТЕЛЬНО КОНВЕРТИРУЕМ В NUMPY
X_train_scaled = np.asarray(X_train_scaled)
X_valid_scaled = np.asarray(X_valid_scaled)

print(f"X_train_scaled тип: {type(X_train_scaled)}")
print(f"X_train_scaled форма: {X_train_scaled.shape}")

# Проверяем на бесконечности (теперь должно работать)
if np.isinf(X_train_scaled).any():
    X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
    X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

alpha = 0.006

lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
lasso.fit(X_train_scaled, y_train_log.values)

selected_mask = np.abs(lasso.coef_) > 1e-5
n_selected = np.sum(selected_mask)
print(f"Alpha = {alpha}")
print(f"Отобрано признаков: {n_selected} из {X_train_scaled.shape[1]}")

X_train_scaled тип: <class 'numpy.ndarray'>
X_train_scaled форма: (956, 246)
Alpha = 0.006
Отобрано признаков: 63 из 246


In [374]:
# # ============================================
# # ПОДБОР ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO
# # ============================================
# print("="*60)
# print("ПОДБОР ОПТИМАЛЬНОГО ALPHA ДЛЯ LASSO")
# print("="*60)

# # Подготавливаем данные (уже есть X_train_processed и X_valid_processed)
# X_train_array = np.array(X_train_processed, dtype=np.float64)
# X_valid_array = np.array(X_valid_processed, dtype=np.float64)

# # Стандартизация
# scaler_lasso = StandardScaler()
# X_train_scaled = scaler_lasso.fit_transform(X_train_array)
# X_valid_scaled = scaler_lasso.transform(X_valid_array)

# # Принудительно конвертируем в numpy
# X_train_scaled = np.asarray(X_train_scaled)
# X_valid_scaled = np.asarray(X_valid_scaled)

# print(f"X_train_scaled форма: {X_train_scaled.shape}")

# # Проверяем на бесконечности
# if np.isinf(X_train_scaled).any():
#     print("⚠️ Найдены бесконечности, заменяем...")
#     X_train_scaled = np.nan_to_num(X_train_scaled, nan=0.0, posinf=0.0, neginf=0.0)
#     X_valid_scaled = np.nan_to_num(X_valid_scaled, nan=0.0, posinf=0.0, neginf=0.0)

# # Разные alpha для тестирования
# alphas = [0.0005, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.008, 0.01, 0.015, 0.02]

# results = []

# print("\nТестирование alpha:")
# print("-" * 70)

# for alpha in alphas:
#     # Обучаем Lasso
#     lasso = Lasso(alpha=alpha, random_state=42, max_iter=10000)
#     lasso.fit(X_train_scaled, y_train_log.values)
    
#     # Отбираем признаки
#     selected_mask = np.abs(lasso.coef_) > 1e-5
#     n_selected = np.sum(selected_mask)
    
#     print(f"alpha={alpha:.4f}: {n_selected} признаков", end="")
    
#     # Если признаков слишком мало, пропускаем
#     if n_selected < 10:
#         print(" (слишком мало, пропускаем)")
#         continue
    
#     # Отбираем данные
#     X_train_selected = X_train_scaled[:, selected_mask]
#     X_valid_selected = X_valid_scaled[:, selected_mask]
    
#     # Быстрое обучение CatBoost
#     cb_temp = CatBoostRegressor(
#         bootstrap_type='Bayesian',
#         iterations=500,
#         learning_rate=0.0279,
#         depth=3,
#         l2_leaf_reg=12.1,
#         bagging_temperature=5.06,
#         random_strength=7.64,
#         colsample_bylevel=0.49,
#         min_data_in_leaf=225,
#         eval_metric='RMSE',
#         loss_function='RMSE',
#         early_stopping_rounds=30,
#         max_bin=128,
#         verbose=False,
#         random_seed=42
#     )
    
#     cb_temp.fit(
#         X_train_selected, 
#         y_train_log.values, 
#         eval_set=(X_valid_selected, y_valid_log.values),
#         verbose=False
#     )
    
#     # Оценка
#     y_pred_valid = cb_temp.predict(X_valid_selected)
#     valid_rmsle = np.sqrt(mean_squared_log_error(
#         np.expm1(y_valid_log.values), 
#         np.expm1(y_pred_valid)
#     ))
    
#     y_pred_train = cb_temp.predict(X_train_selected)
#     train_rmsle = np.sqrt(mean_squared_log_error(
#         np.expm1(y_train_log.values), 
#         np.expm1(y_pred_train)
#     ))
    
#     diff = (valid_rmsle - train_rmsle) / train_rmsle * 100
    
#     print(f", Valid RMSLE={valid_rmsle:.6f}, Разница={diff:.2f}%")
    
#     results.append({
#         'alpha': alpha,
#         'n_features': n_selected,
#         'train_rmsle': train_rmsle,
#         'valid_rmsle': valid_rmsle,
#         'diff': diff
#     })

# # ============================================
# # ВЫВОД РЕЗУЛЬТАТОВ
# # ============================================
# print("\n" + "="*70)
# print("РЕЗУЛЬТАТЫ ПОДБОРА ALPHA")
# print("="*70)

# if results:
#     results_df = pd.DataFrame(results)
#     print(results_df[['alpha', 'n_features', 'valid_rmsle', 'diff']].to_string(index=False))
    
#     # Находим лучший alpha
#     best_result = results_df.loc[results_df['valid_rmsle'].idxmin()]
    
#     print("\n" + "="*70)
#     print(f"🏆 ЛУЧШИЙ ALPHA: {best_result['alpha']:.4f}")
#     print("="*70)
#     print(f"Valid RMSLE: {best_result['valid_rmsle']:.6f}")
#     print(f"Train RMSLE: {best_result['train_rmsle']:.6f}")
#     print(f"Разница: {best_result['diff']:.2f}%")
#     print(f"Отобрано признаков: {int(best_result['n_features'])}")
    
#     # ============================================
#     # ИСПОЛЬЗУЕМ ЛУЧШИЙ ALPHA
#     # ============================================
#     best_alpha = best_result['alpha']
    
#     # Обучаем Lasso с лучшим alpha
#     lasso_best = Lasso(alpha=best_alpha, random_state=42, max_iter=10000)
#     lasso_best.fit(X_train_scaled, y_train_log.values)
    
#     selected_mask_best = np.abs(lasso_best.coef_) > 1e-5
#     n_selected_best = np.sum(selected_mask_best)
    
#     print(f"\n✅ Используем alpha = {best_alpha}")
#     print(f"Отобрано признаков: {n_selected_best} из {X_train_scaled.shape[1]}")
    
#     X_train_selected_best = X_train_scaled[:, selected_mask_best]
#     X_valid_selected_best = X_valid_scaled[:, selected_mask_best]
    
# else:
#     print("⚠️ Нет подходящих результатов, используем alpha = 0.005")
#     best_alpha = 0.005
#     selected_mask_best = np.abs(lasso.coef_) > 1e-5
#     X_train_selected_best = X_train_scaled[:, selected_mask_best]
#     X_valid_selected_best = X_valid_scaled[:, selected_mask_best]

# # ============================================
# # ОБУЧЕНИЕ CATBOOST С ЛУЧШИМ ALPHA
# # ============================================
# print("\n" + "="*70)
# print("ОБУЧЕНИЕ CATBOOST С ЛУЧШИМ ALPHA")
# print("="*70)

# cb = CatBoostRegressor(
#     bootstrap_type='Bayesian',
#     iterations=900,
#     learning_rate=0.0279,
#     depth=3,
#     l2_leaf_reg=12.1,
#     bagging_temperature=5.06,
#     random_strength=7.64,
#     colsample_bylevel=0.49,
#     min_data_in_leaf=225,
#     eval_metric='RMSE',
#     loss_function='RMSE',
#     early_stopping_rounds=44,
#     max_bin=128,
#     verbose=100,
#     random_seed=42
# )

# cb.fit(
#     X_train_selected_best,
#     y_train_log.values,
#     eval_set=(X_valid_selected_best, y_valid_log.values),
#     plot=True
# )

# # ============================================
# # ОЦЕНКА
# # ============================================
# print("\n" + "="*70)
# print("ОЦЕНКА МОДЕЛИ")
# print("="*70)

# y_pred_train_log = cb.predict(X_train_selected_best)
# y_pred_valid_log = cb.predict(X_valid_selected_best)

# y_pred_train_orig = np.expm1(y_pred_train_log)
# y_pred_valid_orig = np.expm1(y_pred_valid_log)
# y_train_orig = np.expm1(y_train_log.values)
# y_valid_orig = np.expm1(y_valid_log.values)

# train_rmse = np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
# valid_rmse = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid_orig))
# train_rmsle = np.sqrt(mean_squared_log_error(y_train_orig, y_pred_train_orig))
# valid_rmsle = np.sqrt(mean_squared_log_error(y_valid_orig, y_pred_valid_orig))

# print(f"Train RMSLE: {train_rmsle:.6f}")
# print(f"Valid RMSLE: {valid_rmsle:.6f}")
# print(f"Разница: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
# print(f"\nTrain RMSE: ${train_rmse:.2f}")
# print(f"Valid RMSE: ${valid_rmse:.2f}")

In [ ]:
X_train_selected = X_train_scaled[:, selected_mask]
X_valid_selected = X_valid_scaled[:, selected_mask]

cb = CatBoostRegressor(
    bootstrap_type='Bayesian',
    iterations=900,
    learning_rate=0.0279,
    depth=3,
    l2_leaf_reg=12.1,
    bagging_temperature=5.06,
    random_strength=7.64,
    colsample_bylevel=0.49,
    min_data_in_leaf=225,
    eval_metric='RMSE',
    loss_function='RMSE',
    early_stopping_rounds=44,
    max_bin=128,
    verbose=100,
    random_seed=42
)

# cb = CatBoostRegressor(
#     cat_features=cat_features,
#     iterations=2000,
#     learning_rate=0.03,
#     depth=5,
#     l2_leaf_reg=12,
#     bagging_temperature=1.0,
#     random_strength=0.7507556902584652,
#     subsample=0.8352580408542853,
#     colsample_bylevel=0.9043302098880885,
#     border_count=64,
#     min_data_in_leaf = 40,
#     eval_metric='RMSE',
#     early_stopping_rounds=100,
#     verbose=100,
#     random_seed=42
# )

cb.fit(
    X_train_selected,
    y_train_log.values,
    eval_set=(X_valid_selected, y_valid_log),
    plot=True
)



MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

0:	learn: 0.3499191	test: 0.3531941	best: 0.3531941 (0)	total: 683us	remaining: 615ms
100:	learn: 0.1518584	test: 0.1701232	best: 0.1701232 (100)	total: 52.7ms	remaining: 417ms
200:	learn: 0.1210911	test: 0.1408152	best: 0.1408152 (200)	total: 127ms	remaining: 442ms
300:	learn: 0.1115251	test: 0.1315997	best: 0.1315997 (300)	total: 184ms	remaining: 367ms
400:	learn: 0.1056770	test: 0.1262152	best: 0.1262152 (400)	total: 237ms	remaining: 295ms
500:	learn: 0.1017730	test: 0.1233960	best: 0.1233960 (500)	total: 293ms	remaining: 234ms
600:	learn: 0.0990088	test: 0.1215936	best: 0.1215936 (600)	total: 345ms	remaining: 172ms
700:	learn: 0.0967036	test: 0.1200028	best: 0.1200028 (700)	total: 406ms	remaining: 115ms
800:	learn: 0.0947297	test: 0.1190276	best: 0.1190263 (797)	total: 533ms	remaining: 65.8ms
899:	learn: 0.0930249	test: 0.1182853	best: 0.1182749 (892)	total: 635ms	remaining: 0us

bestTest = 0.1182749385
bestIteration = 892

Shrink model to first 893 iterations.


CatBoostRegressor(bagging_temperature=5.06, bootstrap_type='Bayesian', colsample_bylevel=0.49, depth=3, early_stopping_rounds=44, eval_metric='RMSE', iterations=900, l2_leaf_reg=12.1, learning_rate=0.0279, loss_function='RMSE', max_bin=128, min_data_in_leaf=225, random_seed=42, random_strength=7.64, verbose=100)

In [376]:
y_pred_train_log = cb.predict(X_train_selected)
y_pred_valid_log = cb.predict(X_valid_selected)

# Преобразуем в оригинальную шкалу
y_pred_train_orig = np.expm1(y_pred_train_log)
y_pred_valid_orig = np.expm1(y_pred_valid_log)

# Оригинальные значения
y_train_orig = np.expm1(y_train_log.values)
y_valid_orig = np.expm1(y_valid_log.values)

In [377]:
train_rmse = np.sqrt(mean_squared_error(y_train_orig, y_pred_train_orig))
train_mae = mean_absolute_error(y_train_orig, y_pred_train_orig)
train_r2 = r2_score(y_train_orig, y_pred_train_orig)

valid_rmse = np.sqrt(mean_squared_error(y_valid_orig, y_pred_valid_orig))
valid_mae = mean_absolute_error(y_valid_orig, y_pred_valid_orig)
valid_r2 = r2_score(y_valid_orig, y_pred_valid_orig)

# RMSLE (главная метрика соревнования)
train_rmsle = np.sqrt(mean_squared_log_error(y_train_orig, y_pred_train_orig))
valid_rmsle = np.sqrt(mean_squared_log_error(y_valid_orig, y_pred_valid_orig))

print("Train Metrics:")
print(f"RMSE: {train_rmse:.4f}")
print(f"MAE: {train_mae:.4f}")
print(f"R²: {train_r2:.4f}")
print("\nValidation Metrics:")
print(f"RMSE: {valid_rmse:.4f}")
print(f"MAE: {valid_mae:.4f}")
print(f"R²: {valid_r2:.4f}")

Train Metrics:
RMSE: 15406.0729
MAE: 10966.8152
R²: 0.9325

Validation Metrics:
RMSE: 17255.1573
MAE: 11789.2377
R²: 0.9119


In [378]:
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error, r2_score

# Предсказания
y_pred_train_log = cb.predict(X_train_selected)
y_pred_valid_log = cb.predict(X_valid_selected)

# Метрики на логарифмической шкале (для RMSLE нужно expm1)
train_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmsle = np.sqrt(mean_squared_log_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

# Дополнительные метрики для понимания
train_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_train_log.values), 
    np.expm1(y_pred_train_log)
))
valid_rmse_orig = np.sqrt(mean_squared_error(
    np.expm1(y_valid_log.values), 
    np.expm1(y_pred_valid_log)
))

print("="*60)
print("МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ")
print("="*60)
print(f"Train RMSLE: {train_rmsle:.6f}")
print(f"Valid RMSLE: {valid_rmsle:.6f}")
print(f"Разница RMSLE: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
print(f"\nRMSE (ориг. шкала):")
print(f"Train: ${train_rmse_orig:.2f}")
print(f"Valid: ${valid_rmse_orig:.2f}")

МЕТРИКИ ДЛЯ СОРЕВНОВАНИЯ
Train RMSLE: 0.093102
Valid RMSLE: 0.118275
Разница RMSLE: 27.04%

RMSE (ориг. шкала):
Train: $15406.07
Valid: $17255.16


In [379]:
# final_pipeline = Pipeline([('preprocessor', preprocessor), ('CatBoost', cb)])
# final_pipeline.fit(X_train, y_train_log)
# test = pd.read_csv('../test.csv')
# result = np.expm1(final_pipeline.predict(test))
# df_result = pd.DataFrame(test['Id'])
# df_result['SalePrice'] = result
# df_result.to_csv('submission.csv', index=False)

In [380]:
# import optuna
# from catboost import CatBoostRegressor
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_error, mean_squared_log_error
# import numpy as np
# import pandas as pd
# import warnings
# warnings.filterwarnings('ignore')

# def objective_catboost_optimized(trial, X, y, cat_indices, n_folds=5):
#     """
#     Оптимизация CatBoost для минимизации переобучения и RMSLE
#     """
#     params = {
#         # Базовые параметры
#         'bootstrap_type': 'Bayesian',
        
#         # Умеренное количество итераций (не слишком много)
#         'iterations': trial.suggest_int('iterations', 700, 950, step=50),
        
#         # Learning rate (ниже = меньше переобучения)
#         'learning_rate': trial.suggest_float('learning_rate', 0.018, 0.028, log=True),
        
#         # Фиксируем глубину 3 (оптимально для этого датасета)
#         'depth': 3,
        
#         # Сильная регуляризация
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 14, 28, log=True),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 220, 380, step=20),
        
#         # Случайность
#         'random_strength': trial.suggest_float('random_strength', 8, 18, log=True),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.45, 0.58),
        
#         # Bayesian параметры
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 5.5, 8.5),
        
#         # Ограничение сложности
#         'max_bin': trial.suggest_int('max_bin', 96, 144, step=16),
        
#         # Ранняя остановка
#         'early_stopping_rounds': trial.suggest_int('early_stopping_rounds', 35, 55),
        
#         # Метрики
#         'eval_metric': 'RMSE',
#         'loss_function': 'RMSE',
#         'verbose': 0,
#         'random_seed': 42,
        
#         # Категориальные признаки
#         'cat_features': cat_indices
#     }
    
#     # Кросс-валидация
#     cv_scores = []
#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         X_tr, X_val = X[train_idx], X[val_idx]
#         y_tr, y_val = y[train_idx], y[val_idx]
        
#         model = CatBoostRegressor(**params)
#         model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False, plot=False)
        
#         # Считаем RMSLE
#         y_pred = model.predict(X_val)
#         rmsle = np.sqrt(mean_squared_log_error(
#             np.expm1(y_val), 
#             np.expm1(y_pred)
#         ))
#         cv_scores.append(rmsle)
        
#         trial.report(np.mean(cv_scores), fold)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
    
#     return np.mean(cv_scores)


# # Подготовка данных
# print("="*60)
# print("ФИНАЛЬНАЯ ОПТИМИЗАЦИЯ")
# print("="*60)

# # Используем 75/25 split (оптимальный баланс)
# X_train, X_valid, y_train_log, y_valid_log = train_test_split(
#     X, y_log, test_size=0.3, random_state=42
# )

# print(f"Размер обучающей: {X_train.shape[0]}")
# print(f"Размер валидационной: {X_valid.shape[0]}")

# # Обучаем препроцессор
# print("\nПрименяем препроцессинг...")
# preprocessor.fit(X_train, y_train_log)

# # Преобразование данных
# X_train_opt = preprocessor.transform(X_train)
# X_valid_opt = preprocessor.transform(X_valid)

# # Находим индексы категориальных колонок
# if hasattr(X_train_opt, 'columns'):
#     cat_indices = [i for i, col in enumerate(X_train_opt.columns) 
#                    if X_train_opt[col].dtype == 'object']
#     X_train_array = X_train_opt.values
#     X_valid_array = X_valid_opt.values
# else:
#     num_count = len([f for f in new_num_features if f not in drop_features])
#     cat_indices = list(range(num_count, num_count + len(cat_features)))
#     X_train_array = X_train_opt
#     X_valid_array = X_valid_opt

# print(f"\nРазмер данных: {X_train_array.shape}")
# print(f"Категориальных колонок: {len(cat_indices)}")

# y_train_array = y_train_log.values
# y_valid_array = y_valid_log.values

# # Запуск оптимизации
# print("\n" + "="*60)
# print("ЗАПУСК ОПТИМИЗАЦИИ")
# print("="*60)

# study = optuna.create_study(
#     direction='minimize',
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_min_trials=10)
# )

# study.optimize(
#     lambda trial: objective_catboost_optimized(
#         trial, X_train_array, y_train_array, cat_indices, n_folds=3
#     ),
#     n_trials=200,
#     show_progress_bar=True
# )

# # Результаты
# print("\n" + "="*60)
# print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
# print("="*60)
# print(f"Лучшее RMSLE (CV): {study.best_value:.6f}")
# print(f"\nЛучшие параметры:")
# for key, value in study.best_params.items():
#     print(f"  {key}: {value}")

# # Обучение финальной модели
# print("\n" + "="*60)
# print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ")
# print("="*60)

# best_params = study.best_params.copy()
# best_params.update({
#     'bootstrap_type': 'Bayesian',
#     'depth': 3,
#     'eval_metric': 'RMSE',
#     'loss_function': 'RMSE',
#     'verbose': 100,
#     'random_seed': 42,
#     'cat_features': cat_indices
# })

# final_model = CatBoostRegressor(**best_params)

# # Обучаем
# final_model.fit(
#     X_train_array,
#     y_train_array,
#     eval_set=(X_valid_array, y_valid_array),
#     verbose=100,
#     plot=True
# )

# # Оценка
# y_pred_train = final_model.predict(X_train_array)
# y_pred_valid = final_model.predict(X_valid_array)

# train_rmsle = np.sqrt(mean_squared_log_error(
#     np.expm1(y_train_array), 
#     np.expm1(y_pred_train)
# ))
# valid_rmsle = np.sqrt(mean_squared_log_error(
#     np.expm1(y_valid_array), 
#     np.expm1(y_pred_valid)
# ))

# train_rmse = np.sqrt(mean_squared_error(y_train_array, y_pred_train))
# valid_rmse = np.sqrt(mean_squared_error(y_valid_array, y_pred_valid))

# print("\n" + "="*60)
# print("ФИНАЛЬНЫЕ МЕТРИКИ")
# print("="*60)
# print(f"Train RMSLE: {train_rmsle:.6f}")
# print(f"Valid RMSLE: {valid_rmsle:.6f}")
# print(f"Разница RMSLE: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
# print(f"\nTrain RMSE: ${np.expm1(train_rmse):.2f}")
# print(f"Valid RMSE: ${np.expm1(valid_rmse):.2f}")

# # Проверка переобучения
# print("\n" + "="*60)
# print("ПРОВЕРКА ПЕРЕОБУЧЕНИЯ")
# print("="*60)
# overfit_pct = (valid_rmsle - train_rmsle) / train_rmsle * 100
# if overfit_pct < 12:
#     print(f"✅ Отлично! Переобучение {overfit_pct:.2f}% (<12%)")
# elif overfit_pct < 18:
#     print(f"✅ Хорошо! Переобучение {overfit_pct:.2f}% (<18%)")
# else:
#     print(f"⚠️ Переобучение {overfit_pct:.2f}%")

# # Топ важных признаков
# print("\n" + "="*60)
# print("ТОП-15 ВАЖНЫХ ПРИЗНАКОВ")
# print("="*60)

# if hasattr(X_train_opt, 'columns'):
#     feature_names = X_train_opt.columns
# else:
#     feature_names = [f'feature_{i}' for i in range(X_train_array.shape[1])]

# feature_importance = pd.DataFrame({
#     'feature': feature_names,
#     'importance': final_model.feature_importances_
# }).sort_values('importance', ascending=False).head(15)

# for idx, row in feature_importance.iterrows():
#     print(f"  {row['feature']}: {row['importance']:.4f}")

# # Создание сабмишна
# print("\n" + "="*60)
# print("СОЗДАНИЕ САБМИШНА")
# print("="*60)

# # Обучаем на ВСЕХ данных
# print("Обучение финальной модели на всех данных...")
# X_all = preprocessor.transform(X)
# y_all = y_log

# final_model_all = CatBoostRegressor(**best_params)
# final_model_all.fit(X_all, y_all, verbose=False)

# # Предсказание
# test = pd.read_csv('../test.csv')
# test_ids = test['Id'].copy()
# test_processed = preprocessor.transform(test)

# if hasattr(test_processed, 'values'):
#     test_array = test_processed.values
# else:
#     test_array = test_processed

# test_pred_log = final_model_all.predict(test_array)
# test_pred = np.expm1(test_pred_log)

# # Сохраняем
# submission = pd.DataFrame({
#     'Id': test_ids,
#     'SalePrice': test_pred
# })
# submission.to_csv('submission_final.csv', index=False)

# print(f"\n✅ Сабмишн сохранен в 'submission_final.csv'")
# print(f"📊 Ожидаемый публичный RMSLE: ~{valid_rmsle + 0.008:.4f}")
# print(f"💰 Ожидаемая ошибка: ~${np.expm1(valid_rmse) + 1000:.0f}")

In [381]:
# import optuna
# from catboost import CatBoostRegressor, Pool
# from sklearn.model_selection import KFold
# from sklearn.metrics import mean_squared_error, mean_squared_log_error
# import numpy as np
# import pandas as pd
# import warnings
# warnings.filterwarnings('ignore')

# def objective_catboost_optimized(trial, X, y, cat_indices, n_folds=5):
#     """
#     Оптимизация CatBoost для минимизации RMSLE и уменьшения переобучения
#     """
#     params = {
#         # Базовые параметры - уменьшаем количество итераций
#         'bootstrap_type': 'Bayesian',
#         'iterations': trial.suggest_int('iterations', 600, 900, step=50),
        
#         # Уменьшаем learning rate для более плавного обучения
#         'learning_rate': trial.suggest_float('learning_rate', 0.015, 0.03, log=True),
        
#         # Минимальная глубина для уменьшения переобучения
#         'depth': trial.suggest_int('depth', 3, 4),
        
#         # Сильная регуляризация
#         'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 15, 30, log=True),
#         'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 150, 300, step=25),
        
#         # Усиленная случайность
#         'random_strength': trial.suggest_float('random_strength', 7, 15, log=True),
#         'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.45, 0.6),
        
#         # Bayesian параметры - больше шума
#         'bagging_temperature': trial.suggest_float('bagging_temperature', 5.0, 8.0),
        
#         # Ограничиваем сложность
#         'max_bin': trial.suggest_int('max_bin', 128, 160, step=32),
        
#         # Дополнительная регуляризация через sampling
#         'subsample': trial.suggest_float('subsample', 0.6, 0.75),
        
#         # Метрики
#         'eval_metric': 'RMSE',
#         'loss_function': 'RMSE',
#         'early_stopping_rounds': 50,
#         'verbose': 0,
#         'random_seed': 42,
        
#         # Категориальные признаки
#         'cat_features': cat_indices
#     }
    
#     # Убираем subsample для Bayesian (несовместимо)
#     if params['bootstrap_type'] == 'Bayesian' and 'subsample' in params:
#         del params['subsample']
    
#     # Кросс-валидация
#     cv_scores = []
#     kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    
#     for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
#         X_tr, X_val = X[train_idx], X[val_idx]
#         y_tr, y_val = y[train_idx], y[val_idx]
        
#         model = CatBoostRegressor(**params)
#         model.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False, plot=False)
        
#         # Считаем RMSLE на валидации
#         y_pred = model.predict(X_val)
#         rmsle = np.sqrt(mean_squared_log_error(
#             np.expm1(y_val), 
#             np.expm1(y_pred)
#         ))
#         cv_scores.append(rmsle)
        
#         trial.report(np.mean(cv_scores), fold)
#         if trial.should_prune():
#             raise optuna.TrialPruned()
    
#     return np.mean(cv_scores)


# # Подготовка данных
# print("Подготовка данных для оптимизации...")

# # Обучаем препроцессор
# preprocessor.fit(X_train, y_train_log)

# # Преобразование данных
# X_train_opt = preprocessor.transform(X_train)
# X_valid_opt = preprocessor.transform(X_valid)

# # Находим индексы категориальных колонок
# if hasattr(X_train_opt, 'columns'):
#     cat_indices = [i for i, col in enumerate(X_train_opt.columns) if X_train_opt[col].dtype == 'object']
#     X_train_array = X_train_opt.values
#     X_valid_array = X_valid_opt.values
# else:
#     num_count = len([f for f in new_num_features if f not in drop_features])
#     cat_indices = list(range(num_count, num_count + len(cat_features)))
#     X_train_array = X_train_opt
#     X_valid_array = X_valid_opt

# print(f"Размер данных: {X_train_array.shape}")
# print(f"Категориальных колонок: {len(cat_indices)}")

# y_train_array = y_train_log.values
# y_valid_array = y_valid_log.values

# # Запуск оптимизации
# print("\nЗапуск оптимизации CatBoost (минимизация RMSLE)...")
# study = optuna.create_study(
#     direction='minimize',
#     sampler=optuna.samplers.TPESampler(seed=42),
#     pruner=optuna.pruners.MedianPruner(n_warmup_steps=5, n_min_trials=10)
# )

# study.optimize(
#     lambda trial: objective_catboost_optimized(
#         trial, X_train_array, y_train_array, cat_indices, n_folds=3
#     ),
#     n_trials=200,
#     show_progress_bar=True
# )

# print("\n" + "="*60)
# print("РЕЗУЛЬТАТЫ ОПТИМИЗАЦИИ")
# print("="*60)
# print(f"Лучшее RMSLE (CV): {study.best_value:.6f}")
# print(f"\nЛучшие параметры для уменьшения переобучения:")
# for key, value in study.best_params.items():
#     print(f"  {key}: {value}")

# # Обучение финальной модели
# print("\n" + "="*60)
# print("ОБУЧЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ")
# print("="*60)

# best_params = study.best_params.copy()
# best_params.update({
#     'bootstrap_type': 'Bayesian',
#     'eval_metric': 'RMSE',
#     'loss_function': 'RMSE',
#     'early_stopping_rounds': 50,
#     'verbose': 100,
#     'random_seed': 42,
#     'cat_features': cat_indices
# })

# # Убираем subsample если есть
# if 'subsample' in best_params:
#     del best_params['subsample']

# final_model = CatBoostRegressor(**best_params)

# # Обучаем
# final_model.fit(
#     X_train_array,
#     y_train_array,
#     eval_set=(X_valid_array, y_valid_array),
#     verbose=100,
#     plot=True
# )

# # Оценка RMSLE
# y_pred_train = final_model.predict(X_train_array)
# y_pred_valid = final_model.predict(X_valid_array)

# train_rmsle = np.sqrt(mean_squared_log_error(
#     np.expm1(y_train_array), 
#     np.expm1(y_pred_train)
# ))
# valid_rmsle = np.sqrt(mean_squared_log_error(
#     np.expm1(y_valid_array), 
#     np.expm1(y_pred_valid)
# ))

# train_rmse = np.sqrt(mean_squared_error(y_train_array, y_pred_train))
# valid_rmse = np.sqrt(mean_squared_error(y_valid_array, y_pred_valid))

# print("\n" + "="*60)
# print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ")
# print("="*60)
# print(f"Train RMSLE: {train_rmsle:.6f}")
# print(f"Valid RMSLE: {valid_rmsle:.6f}")
# print(f"Разница: {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")
# print(f"\nTrain RMSE (log): {train_rmse:.6f}")
# print(f"Valid RMSE (log): {valid_rmse:.6f}")

# # Проверка переобучения
# print("\n" + "="*60)
# print("ПРОВЕРКА ПЕРЕОБУЧЕНИЯ")
# print("="*60)
# if (valid_rmsle - train_rmsle) / train_rmsle < 0.10:
#     print("✅ Отлично! Переобучение < 10%")
# elif (valid_rmsle - train_rmsle) / train_rmsle < 0.15:
#     print("✅ Хорошо! Переобучение < 15%")
# else:
#     print(f"⚠️ Переобучение {((valid_rmsle - train_rmsle) / train_rmsle * 100):.2f}%")

# # Сравнение с вашими текущими результатами
# print("\n" + "="*60)
# print("СРАВНЕНИЕ С ТЕКУЩЕЙ МОДЕЛЬЮ")
# print("="*60)
# print(f"Текущий Valid RMSLE: 0.116964")
# print(f"Новый Valid RMSLE: {valid_rmsle:.6f}")
# print(f"Улучшение: {(0.116964 - valid_rmsle) * 1000:.1f} пунктов RMSLE")

# # Важность признаков
# print("\n" + "="*60)
# print("ТОП-10 ВАЖНЫХ ПРИЗНАКОВ")
# print("="*60)

# if hasattr(X_train_opt, 'columns'):
#     feature_names = X_train_opt.columns
# else:
#     feature_names = [f'feature_{i}' for i in range(X_train_array.shape[1])]

# feature_importance = pd.DataFrame({
#     'feature': feature_names,
#     'importance': final_model.feature_importances_
# }).sort_values('importance', ascending=False).head(10)

# for idx, row in feature_importance.iterrows():
#     print(f"  {row['feature']}: {row['importance']:.4f}")

# # Сохраняем параметры
# import json
# with open('best_params_rmsle.json', 'w') as f:
#     json.dump(study.best_params, f, indent=4)
# print("\n✅ Лучшие параметры сохранены в 'best_params_rmsle.json'")